# Construction de la scène Tchaikovsky
Spatialisation binaurale de chaque stem de l'orchestre depuis le dataset The Spheres.

In [ ]:
import sys, os
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

## 1. Initialisation du spatializer

In [ ]:
from src.instrument import InstrumentSpatializer, ListenerConfig, RenderConfig

spatializer = InstrumentSpatializer(
    hrtf_path           = 'dataset/generic.sofa',
    positions_json      = 'dataset_live/positions_phase1.json',
    channel_map_json    = 'dataset_live/channel_map_Tchaikovsky.json',
    mic_folder_map_json = 'dataset_live/mic_folder_map_Tchaikovsky.json',
    dataset_root        = 'dataset_live',
    piece               = 'Tchaikovsky',
    config              = RenderConfig(normalize=False),
)

## 2. Diagnostic dataset — structure The Spheres

In [ ]:
# Cellule de diagnostic : le dossier = micro, le fichier = source
# Violin_1/Trumpet_1.flac  → sonne trompette  (micro violon capte la trompette)
# Trumpet/Violin_1_1.flac  → sonne violon     (micro trompette capte le violon)
import soundfile as sf
from IPython.display import Audio, display

print('Violin_1/ -> Violin_1_1.flac  (micro natif — doit sonner violon)')
data, sr = sf.read(ROOT / 'dataset_live/Tchaikovsky/Violin_1/Violin_1_1.flac')
display(Audio(data, rate=sr))

In [ ]:
print('Violin_1/ -> Trumpet_1.flac  (micro trompette dans dossier violon)')
data, sr = sf.read(ROOT / 'dataset_live/Tchaikovsky/Violin_1/Trumpet_1.flac')
display(Audio(data, rate=sr))

In [ ]:
print('Trumpet/ -> Violin_1_1.flac  (micro violon dans dossier trompette)')
data, sr = sf.read(ROOT / 'dataset_live/Tchaikovsky/Trumpet/Violin_1_1.flac')
display(Audio(data, rate=sr))

In [ ]:
print('Trumpet/ -> Trumpet_1.flac  (micro natif — doit sonner trompette)')
data, sr = sf.read(ROOT / 'dataset_live/Tchaikovsky/Trumpet/Trumpet_1.flac')
display(Audio(data, rate=sr))

## 3. Test sur Bassoon — render_instrument (2 stems → 2 WAV)

In [ ]:
results = spatializer.render_instrument('Bassoon', output_dir='sound/scene/')
# -> sound/scene/Bassoon_1_spatial.wav
# -> sound/scene/Bassoon_2_spatial.wav

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

display(Audio(filename=str(ROOT / 'sound/scene/Bassoon_1_spatial.wav')))

In [ ]:
display(Audio(filename=str(ROOT / 'sound/scene/Bassoon_2_spatial.wav')))

## 4. Génération de tous les stems de l'orchestre (long — ~95 min)

In [ ]:
import soundfile as sf
import numpy as np
from pathlib import Path

INSTRUMENTS = spatializer.available_instruments()
print(f'{len(INSTRUMENTS)} instruments : {INSTRUMENTS}\n')

OUT_DIR = Path('sound/tchaikovsky_scene')

for instrument in INSTRUMENTS:
    print(f"\n{'='*60}")
    print(f'Instrument : {instrument}')
    try:
        spatializer.render_instrument(instrument, output_dir=OUT_DIR, normalize=False)
    except Exception as e:
        print(f'  [ERREUR] {instrument} : {e}')

## 5. Mix final — somme de tous les stems

In [ ]:
from src.instrument import InstrumentSpatializer
import soundfile as sf
from pathlib import Path

OUT_DIR = Path('sound/tchaikovsky_scene')
stem_files = sorted(OUT_DIR.glob('*_spatial.wav'))
print(f'{len(stem_files)} stems à mixer\n')

signals, sr_final = [], None
for f in stem_files:
    data, sr = sf.read(str(f), dtype='float32', always_2d=True)
    signals.append(data)
    sr_final = sr
    print(f'  + {f.name}')

mix  = InstrumentSpatializer.mix_stereo(signals)
peak = max(abs(mix.max()), abs(mix.min())) + 1e-10
mix  = (mix / peak).astype('float32')

sf.write('sound/tchaikovsky_full.wav', mix, sr_final)
print('\nMix final -> sound/tchaikovsky_full.wav')

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
display(Audio(filename=str(ROOT / 'sound/tchaikovsky_full.wav')))